**Title:** LRR Case Study - Part 2 (Company Crosswalk & Entity Resolution)<br>
**Created By:** Begimay Ahmatova<br>
**Created Date:** 2026-10-02<br>
**Description:** Notebook resolves account.csv against final companies table.<br>

In [22]:
## libraries
import difflib
import re

import duckdb as db
import numpy as np
import pandas as pd

In [23]:
cols = [
    "id", "name", "billingcity", "billingstate", "billingcountry",
    "phone", "website", "domain__c",
    "ss_company_name__c", "ss_website__c", "ss_city__c",
    "ss_state__c", "ss_phone__c", "ss_year_founded__c",
]

accounts = pd.read_csv("account.csv", usecols=cols)[cols]


In [24]:
accounts = accounts.rename(columns={"domain__c": "domain"})

In [25]:
accounts["year_founded"] = accounts["ss_year_founded__c"].astype("Int64")

In [26]:
digits = accounts["phone"].astype("string").str.replace(r"\D", "", regex=True)

na_countries = {"united states", "canada"}
is_na = accounts["billingcountry"].astype("string").str.strip().str.lower().isin(na_countries)

m10 = (digits.str.len() == 10) & is_na
m11 = (digits.str.len() == 11) & digits.str.startswith("1")

accounts["phone_std"] = accounts["phone"]
accounts.loc[m10, "phone_std"] = "+1" + digits[m10]
accounts.loc[m11, "phone_std"] = "+" + digits[m11]

print(m10.sum(), "+1 added;", m11.sum(), "already had the 1")


68 +1 added; 26 already had the 1


In [27]:
digits = accounts["ss_phone__c"].astype("string").str.replace(r"\D", "", regex=True)

na_countries = {"united states", "canada"}
is_na = accounts["billingcountry"].astype("string").str.strip().str.lower().isin(na_countries)

m10 = (digits.str.len() == 10) & is_na
m11 = (digits.str.len() == 11) & digits.str.startswith("1")

accounts["ss_phone_std"] = accounts["ss_phone__c"]
accounts.loc[m10, "ss_phone_std"] = "+1" + digits[m10]
accounts.loc[m11, "ss_phone_std"] = "+" + digits[m11]

print(m10.sum(), "+1 added;", m11.sum(), "already had the 1")


36 +1 added; 43 already had the 1


In [28]:
def norm_domain(x):
    """lowercase, drop scheme / www. / path -> bare domain (None if blank)"""
    if pd.isna(x):
        return None
    x = re.sub(r"^https?://", "", str(x).strip().lower())
    x = re.sub(r"^www\.", "", x).split("/")[0]
    return x or None

accounts["domain_std"] = accounts["domain"].map(norm_domain)
accounts["ss_domain_std"] = accounts["ss_website__c"].map(norm_domain)
accounts["website_std"] = accounts["website"].map(norm_domain)

print((accounts["domain"].notna() & (accounts["domain"] != accounts["domain_std"])).sum(), "domain values cleaned")

24 domain values cleaned


In [29]:
accounts.head()

,id,name,billingcity,billingstate,billingcountry,phone,website,domain,ss_company_name__c,ss_website__c,ss_city__c,ss_state__c,ss_phone__c,ss_year_founded__c,year_founded,phone_std,ss_phone_std,domain_std,ss_domain_std,website_std
0,00157Q1WAHSCDPHCUN,"Rare Earth Exploration, LLC",Garland,Texas,United States,(469) 480-3141,https://rareearthex.com,rareearthex.com,"Rare Earth Exploration, LLC",https://rareearthex.com,Garland,Texas,+14694803141,2016.0,2016,+14694803141,+14694803141,rareearthex.com,rareearthex.com,rareearthex.com
1,001R7G4RI0GA09H5ZJ,Kenton Grant Consulting,Tulsa,Oklahoma,United States,+19188123774,https://www.kentongrantconsulting.com,kentongrantconsulting.com,Kenton Grant Consulting,https://kentongrantconsulting.com,Tulsa,Oklahoma,+19188123774,2014.0,2014,+19188123774,+19188123774,kentongrantconsulting.com,kentongrantconsulting.com,kentongrantconsulting.com
2,0012ABVJ564CCELZ4K,"GUTHY-RENKER, LLC",El Segundo,California,United States,888-651-6602,https://guthy-renker.com,guthy-renker.com,"Guthy-Renker, LLC",https://guthy-renker.com,El Segundo,California,888-651-6602,1988.0,1988,+18886516602,+18886516602,guthy-renker.com,guthy-renker.com,guthy-renker.com
3,001R0ST0DTW00BXMZZ,SOS TRUCK SERVICE,Ontario,California,United States,(888) 999-9767,NaN,NaN,SOS TRUCK SERVICE,NaN,Ontario,California,+18889999767,1967.0,1967,+18889999767,+18889999767,NaN,NaN,NaN
4,00186JM0HJK76GBGEK,Nutrition Partners Inc.,Airdrie,Alberta,Canada,403-912-0735,https://nutritionpartners.ca,nutritionpartners.ca,"Nutrition Partners, Inc.",https://nutritionpartners.ca,Airdrie,Alberta,+14039120735,2003.0,2003,+14039120735,+14039120735,nutritionpartners.ca,nutritionpartners.ca,nutritionpartners.ca


In [30]:
con = db.connect("llr.duckdb")

In [31]:
# only pull the columns you need
company = con.sql("""
    SELECT *
    FROM deduped_company
""").df()

In [32]:
company.head()

,id,normalized_name,informalname,cleaned_informalname,website,domain,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date
0,9700,Medi-Code,Medi-Code,medi-code,http://medi-code.com,medi-code.com,+18324372029,Katy,Texas,77494,United States,2009,1,<NA>,NaN,<NA>,False,2026-09-15
1,20074,Marketware,Marketware,marketware,https://marketware.com,marketware.com,+18019444230,Sandy,Utah,84094,United States,2014,4,1665037,Medsphere Systems Corporation,5900000,False,2026-07-16
2,28462,VCNetwork,VCNetwork,vcnetwork,https://vcnetwork.org,vcnetwork.org,,San Francisco,California,94114,United States,1997,5,<NA>,NaN,<NA>,False,2026-09-11
3,35837,Complipoint,Complipoint,complipoint,https://www.complipoint.com,complipoint.com,+18586732174,San Diego,California,92127,United States,2002,8,204227,Applications International Corp. (AIC),<NA>,False,2025-11-26
4,45311,FlexiInternational Software,FlexiInternational Software,flexiinternational software,https://www.flexi.com,flexi.com,+18003539492,Shelton,Connecticut,06484,United States,1992,3,<NA>,NaN,<NA>,False,2026-08-08


In [33]:
# Part 1 side of the crosswalk: every raw company id (1,024) and where it ended up after dedupe
raw_company = con.sql("SELECT id, name FROM raw_company").df()
company_map = con.sql("SELECT raw_id, final_id FROM company_map").df()
print(len(raw_company), "raw companies ->", company_map["final_id"].nunique(), "final companies")

1024 raw companies -> 999 final companies


In [34]:
# ---------- helpers ----------
LEGAL = {"inc", "llc", "ltd", "corp", "corporation", "co", "company", "lp", "llp",
         "pllc", "pc", "the", "and", "dba", "fka"}

def name_tokens(s):
    toks = re.findall(r"[a-z0-9]+", str(s).lower()) if pd.notna(s) else []
    return [t for t in toks if t not in LEGAL]

def name_sim(a, b):
    """0-1 name similarity. 1.0 if one name's words are all inside the other (e.g. 'X (fka Y)' vs 'Y')."""
    ta, tb = name_tokens(a), name_tokens(b)
    if not ta or not tb:
        return 0.0
    if set(ta) <= set(tb) or set(tb) <= set(ta):
        return 1.0
    return difflib.SequenceMatcher(None, " ".join(ta), " ".join(tb)).ratio()

def unique_map(keys, ids):
    """key -> company id, dropping any key shared by more than one company (ambiguous)."""
    df = pd.DataFrame({"k": keys, "id": ids}).dropna(subset=["k"])
    df = df[df["k"] != ""].drop_duplicates()
    return df[~df["k"].duplicated(keep=False)].set_index("k")["id"].to_dict()

# ---------- lookups against the final Part 1 companies ----------
dom_map = unique_map(company["domain"], company["id"])   # company domains are already clean
ph_map = unique_map(company["normalized_phone"], company["id"])
co = company.set_index("id")

DOMAIN_FIELDS = ["domain_std", "ss_domain_std", "website_std"]   # cleaned from domain__c, ss_website__c, website
PHONE_FIELDS = ["phone_std", "ss_phone_std"]

def same_place(acct_vals, co_val):
    """True/False if we can compare, None if either side is blank."""
    a = {str(v).strip().lower() for v in acct_vals if pd.notna(v)}
    if not a or pd.isna(co_val):
        return None
    return str(co_val).strip().lower() in a

LEVELS = ["low", "medium", "high"]

def match_account(r):
    # 1) candidates: any company that shares a domain or phone with the account
    cands = {}
    for f in DOMAIN_FIELDS:
        cid = dom_map.get(getattr(r, f))
        if cid is not None:
            cands.setdefault(cid, {"domain": [], "phone": []})["domain"].append(f)
    for f in PHONE_FIELDS:
        cid = ph_map.get(getattr(r, f)) if pd.notna(getattr(r, f)) else None
        if cid is not None:
            cands.setdefault(cid, {"domain": [], "phone": []})["phone"].append(f)
    if not cands:
        return None

    # 2) score each candidate
    scored = []
    for cid, sig in cands.items():
        c = co.loc[cid]
        co_names = (c["normalized_name"], c["informalname"])
        sim_name = max(name_sim(r.name, cn) for cn in co_names)
        sim_ss = max(name_sim(r.ss_company_name__c, cn) for cn in co_names)
        sim = max(sim_name, sim_ss)
        city = same_place([r.billingcity], c["city"])      # billing address is the account's own field
        state = same_place([r.billingstate], c["state"])
        ss_city = same_place([r.ss_city__c], c["city"])    # ss_* = enrichment, used as a cross-check
        ss_state = same_place([r.ss_state__c], c["state"])
        score = 3 * bool(sig["domain"]) + 3 * bool(sig["phone"]) + 2 * (sim >= 0.8) + 0.5 * (city is True) + 0.5 * (state is True)
        scored.append(dict(cid=cid, sig=sig, score=score, sim=sim, sim_name=sim_name, sim_ss=sim_ss, city=city, state=state,
                           ss_agrees=(ss_city is True or ss_state is True)))
    scored.sort(key=lambda x: -x["score"])
    best = scored[0]
    if len(scored) > 1 and scored[1]["score"] == best["score"]:
        return dict(company_id=None, confidence=None, basis=None,
                    flags="ambiguous: tied candidates " + ", ".join(co.loc[s["cid"], "normalized_name"] for s in scored[:2]))

    # 3) basis + confidence
    c = co.loc[best["cid"]]
    basis = [s for s in ("domain", "phone") if best["sig"][s]]
    if best["sim"] >= 0.8:
        basis.append("name")
    if best["city"]:
        basis.append("city")
    if best["state"]:
        basis.append("state")
    n_strong = len([b for b in basis if b in ("domain", "phone", "name")])
    level = 2 if n_strong >= 2 else (1 if best["sim"] >= 0.5 else 0)
    flags = []
    if best["sim_name"] < 0.5:
        if best["sim_ss"] >= 0.8:
            flags.append(f"account name '{r.name}' differs from '{c['normalized_name']}' but ss_company_name__c agrees (likely rename)")
        else:
            flags.append(f"name differs: '{r.name}' vs '{c['normalized_name']}'")
    if best["city"] is False or best["state"] is False:
        level -= 1
        note = " (ss_ fields agree with the company)" if best["ss_agrees"] else ""
        flags.append(f"billing location differs: {r.billingcity}, {r.billingstate} vs {c['city']}, {c['state']}{note}")
    if len(scored) > 1:
        level = min(level, 1)
        others = "; ".join(f"{'+'.join(k for k in ('domain', 'phone') if s['sig'][k])} -> {co.loc[s['cid'], 'normalized_name']} ({s['cid']})" for s in scored[1:])
        flags.append(f"conflict: other signals point elsewhere: {others}")
    return dict(company_id=best["cid"], confidence=LEVELS[max(level, 0)],
                basis="+".join(basis), flags="; ".join(flags) or None)

# ---------- run it ----------
res = [match_account(r) for r in accounts.itertuples()]
matches = pd.DataFrame([x or dict(company_id=None, confidence=None, basis=None, flags=None) for x in res], index=accounts.index)
matches["company_id"] = matches["company_id"].astype("Int64")

# ---------- unmatched accounts: flag near-misses instead of forcing a match ----------
claimed = set(matches["company_id"].dropna())
for i in matches.index[matches["company_id"].isna()]:
    r = accounts.loc[i]
    sims = company["normalized_name"].map(lambda n: name_sim(r["name"], n))
    same_lead = company["normalized_name"].map(lambda n: name_tokens(n)[:1] == name_tokens(r["name"])[:1])
    sims = sims.where(same_lead, 0)          # must at least share the first word, else it's just generic words
    j = sims.idxmax()
    if sims[j] >= 0.7:
        taken = " (that company is already matched to another account)" if company.loc[j, "id"] in claimed else ""
        matches.loc[i, "flags"] = f"possible near-miss, NOT matched (name only, no domain/phone): '{company.loc[j, 'normalized_name']}' ({company.loc[j, 'id']}), sim {sims[j]:.2f}{taken}"

accounts = accounts.drop(columns=[c for c in matches.columns if c in accounts.columns]).join(matches)
accounts["confidence"].value_counts(dropna=False)

confidence
high      84
NaN       10
medium     6
Name: count, dtype: int64

In [35]:
SYS_P1, SYS_SF = "part1_company", "salesforce"

# Part 1 rows: one per RAW company id (keeps merge lineage); grouped by the final company they merged into
p1 = raw_company.merge(company_map, left_on="id", right_on="raw_id", how="left")
assert p1["final_id"].notna().all()
p1 = pd.DataFrame({
    "source_system": SYS_P1,
    "source_system_id": p1["id"].astype(str),
    "company_name": p1["name"],
    "entity": "p1:" + p1["final_id"].astype(str),
})

# Salesforce rows: one per account; matched ones join their Part 1 company, unmatched stay on their own
sf = pd.DataFrame({
    "source_system": SYS_SF,
    "source_system_id": accounts["id"].astype(str),
    "company_name": accounts["name"],
    "entity": np.where(accounts["company_id"].notna(), "p1:" + accounts["company_id"].astype(str), "sf:" + accounts["id"]),
    "match_confidence": accounts["confidence"],
    "match_basis": accounts["basis"],
    "flags": accounts["flags"],
})

xw = pd.concat([p1, sf], ignore_index=True)

# crosswalk_id: one per real-world company, shared by every record in it (Part 1 companies first, then Salesforce-only)
order = sorted(xw["entity"].unique(), key=lambda e: (e.startswith("sf:"), int(e[3:]) if e.startswith("p1:") else 0, e))
xw["crosswalk_id"] = xw["entity"].map({e: i + 1 for i, e in enumerate(order)})

# presence: which systems each company is in (works for any number of sources; a 3rd source = just more rows)
systems = xw.groupby("crosswalk_id")["source_system"].agg(set)
def presence(s):
    if {SYS_P1, SYS_SF} <= s: return "both"
    return "part1_only" if s == {SYS_P1} else "salesforce_only"
xw["presence"] = xw["crosswalk_id"].map(systems.map(presence))

# a match belongs to the company, so every row of a matched company carries its confidence/basis/flags
rank = {"low": 0, "medium": 1, "high": 2}
m = xw[xw["match_confidence"].notna()].groupby("crosswalk_id").agg(
    match_confidence=("match_confidence", lambda s: min(s, key=rank.get)),
    match_basis=("match_basis", lambda s: " | ".join(dict.fromkeys(s))),
)
fl = xw[xw["flags"].notna()].groupby("crosswalk_id")["flags"].agg(lambda s: " | ".join(dict.fromkeys(s)))
xw = xw.drop(columns=["match_confidence", "match_basis", "flags"])
xw = xw.merge(m, on="crosswalk_id", how="left").merge(fl, on="crosswalk_id", how="left")
xw.loc[xw["presence"] != "both", ["match_confidence", "match_basis"]] = None   # blank on single-system rows

crosswalk = (xw[["crosswalk_id", "source_system", "source_system_id", "company_name",
                 "presence", "match_confidence", "match_basis", "flags"]]
             .sort_values(["crosswalk_id", "source_system", "source_system_id"]).reset_index(drop=True))
crosswalk.head(10)

,crosswalk_id,source_system,source_system_id,company_name,presence,match_confidence,match_basis,flags
0,1,part1_company,9700,"Medi-Code, LLC",part1_only,NaN,NaN,NaN
1,2,part1_company,20074,"Marketware, Inc.",part1_only,NaN,NaN,NaN
2,3,part1_company,28462,VCNetwork,part1_only,NaN,NaN,NaN
3,4,part1_company,35837,"Complipoint, LLC",part1_only,NaN,NaN,NaN
4,5,part1_company,45311,"FlexiInternational Software, Inc.",part1_only,NaN,NaN,NaN
5,6,part1_company,48678,"Blackhawk Specialty Tools, LLC",part1_only,NaN,NaN,NaN
6,7,part1_company,67565,"DemandFarm, Inc.",part1_only,NaN,NaN,NaN
7,8,part1_company,71013,"Sutoer Solutions, Inc.",part1_only,NaN,NaN,NaN
8,9,part1_company,83110,"Implantech Associates, Inc.",part1_only,NaN,NaN,NaN
9,10,part1_company,84414,Dorado Industries,both,high,phone+name+city+state,NaN


In [36]:
con.sql("CREATE OR REPLACE TABLE company_crosswalk AS SELECT * FROM crosswalk")

In [37]:
# ---- completeness checks ----
assert len(crosswalk) == len(raw_company) + len(accounts), "row count"
assert set(crosswalk.loc[crosswalk.source_system == "part1_company", "source_system_id"]) == set(raw_company["id"].astype(str)), "raw ids"
assert set(crosswalk.loc[crosswalk.source_system == "salesforce", "source_system_id"]) == set(accounts["id"].astype(str)), "account ids"
assert not crosswalk.duplicated(["source_system", "source_system_id"]).any(), "duplicates"
print(len(crosswalk), "rows =", (crosswalk.source_system == "part1_company").sum(), "part1 +", (crosswalk.source_system == "salesforce").sum(), "salesforce\n")

print("companies by presence (one per crosswalk_id):")
print(crosswalk.groupby("presence")["crosswalk_id"].nunique().to_string(), "\n")
print("match confidence (companies in both):")
print(crosswalk[crosswalk.presence == "both"].drop_duplicates("crosswalk_id")["match_confidence"].value_counts().to_string(), "\n")

# everything flagged, one line per company
flagged = crosswalk[crosswalk["flags"].notna()].drop_duplicates("crosswalk_id")
flagged[["crosswalk_id", "presence", "company_name", "match_confidence", "match_basis", "flags"]]

1124 rows = 1024 part1 + 100 salesforce

companies by presence (one per crosswalk_id):
presence
both                90
part1_only         909
salesforce_only     10 

match confidence (companies in both):
match_confidence
high      84
medium     6 



,crosswalk_id,presence,company_name,match_confidence,match_basis,flags
94,88,both,VyMaC Corp.,medium,domain+phone+name+city+state,conflict: other signals point elsewhere: domai...
160,146,both,Wyoming Mutual Telephone Co.,medium,domain+phone+name+city+state,conflict: other signals point elsewhere: domai...
173,158,both,Sanskar Culture Society,medium,domain+phone+name,"billing location differs: Providence, Rhode Is..."
207,189,both,"Odd13 Brewing, Inc.",high,domain+phone+city+state,name differs: 'Thirteen Barrels Brewing Co.' v...
234,211,both,"Haeger, Inc.",medium,domain+phone+name+state,"billing location differs: Grand Rapids, Michig..."
734,676,both,ALIGNMT LLC,medium,domain+phone+name+city+state,conflict: other signals point elsewhere: domai...
907,834,both,Bateman Labs,medium,domain+phone+name,"billing location differs: Calgary, Alberta vs ..."
1097,989,both,Centrum-AI Inc.,high,domain+name+city+state,account name 'Helix Intelligence Group' differ...
1114,1000,salesforce_only,Bateman Laboratories Inc.,NaN,NaN,"possible near-miss, NOT matched (name only, no..."
1123,1009,salesforce_only,Dorado Industrial Group,NaN,NaN,"possible near-miss, NOT matched (name only, no..."


**Write-Up**<br>
<br>
**1. Matching order and field reliability**<br>
I matched on domain first, since it has the fewest nulls of the matching fields in accounts and was the key used in Part 1, after removing http/www prefixes; because no single field should decide a match, I also used standardized phone numbers. Any two of domain, phone and a fuzzy name match made a high-confidence match, a city/state mismatch or a conflicting signal lowered it to medium, and ties were left unmatched and flagged for manual review.

**2. Unmatched records and avoiding false matches**<br>
Ten accounts stayed unmatched because no domain or phone matched; two of them (Dorado Industrial Group, Bateman Laboratories) look like existing companies, but I flagged them instead of matching on name alone. I went back and forth on Helix Intelligence Group → Centrum-AI and Thirteen Barrels → Odd13 Brewing, and kept both because domain and phone agree.

**3. Scaling to 100k+ records**<br>
I think there is room for improvement on the name fuzzy matching. I'd also give each company a stable crosswalk_id that never changes, and store a hash of the tracked fields (name, domain, phone) so that when a known record changes it is updated in place instead of being matched again as a new record.

**4. Low-confidence matches in production**<br>
High-confidence and medium matches would be auto-accepted, and accounts with no domain or phone hit would stay unmatched rather than be rejected. Low and conflicting matches would go to a review queue owned by the team that manages Salesforce account data, and their decisions would be saved so the same pair isn't reviewed twice.

In [38]:
con.close()